# 07 - Topic Sentiment Analysis and Reporting (Objective 5 + reporting)
1. Score every **sentence** with the fine-tuned BERT model.
2. Aggregate sentiment by topic/theme, hotel and month.
3. Store results in a **separate table** linked by `review_id` (never write into source tables).

In [ ]:
%pip install -q transformers torch pandas seaborn matplotlib pyarrow sqlalchemy

In [ ]:
import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import pipeline
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
sent = pd.read_parquet("data/processed/sentences_topics.parquet")
reviews = pd.read_parquet("data/processed/reviews_clean.parquet")

clf = pipeline("text-classification", model="models/bert_sentiment", tokenizer="models/bert_sentiment",
               top_k=None, truncation=True, max_length=128, batch_size=64,
               device=0 if torch.cuda.is_available() else -1)

## 1. Sentiment for every sentence

In [ ]:
raw = clf(sent["sentence"].tolist())          # list of [{label, score}, ...] per sentence
probs = pd.DataFrame([{d["label"]: d["score"] for d in row} for row in raw])

sent = pd.concat([sent.reset_index(drop=True), probs], axis=1)
sent["sentiment"] = probs[["negative", "neutral", "positive"]].idxmax(axis=1)
sent["sentiment_score"] = sent["positive"] - sent["negative"]      # -1 (very negative) ... +1 (very positive)
sent[["sentence", "theme", "sentiment", "sentiment_score"]].head()

## 2. Which topics are positive or negative?

In [ ]:
topic_df = sent[sent["theme"].notna() & ~sent["theme"].isin(["outlier", "other"])]

share = pd.crosstab(topic_df["theme"], topic_df["sentiment"], normalize="index")[["negative", "neutral", "positive"]]
share.sort_values("positive").mul(100).plot.barh(stacked=True, color=["#d9534f", "#f0ad4e", "#5cb85c"], figsize=(9, 4))
plt.xlabel("% of sentences"); plt.title("Sentiment share by topic"); plt.legend(loc="lower right"); plt.show()

In [ ]:
# hotel x topic heatmap of average sentiment
hotel_names = reviews.drop_duplicates("hotel_id").set_index("hotel_id")["hotel_name"]
topic_df = topic_df.assign(hotel=topic_df["hotel_id"].map(hotel_names))
heat = topic_df.pivot_table(index="hotel", columns="theme", values="sentiment_score", aggfunc="mean")

plt.figure(figsize=(10, 5))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn", center=0)
plt.title("Average sentiment score: hotel x topic"); plt.show()

In [ ]:
# trend over time
topic_df = topic_df.assign(month=pd.to_datetime(topic_df["review_date"]).dt.to_period("M").dt.to_timestamp())
trend = topic_df.pivot_table(index="month", columns="theme", values="sentiment_score", aggfunc="mean")
trend.rolling(3, min_periods=1).mean().plot(figsize=(11, 4), title="Sentiment trend by topic (3-month rolling)")
plt.ylabel("avg sentiment"); plt.show()

In [ ]:
# top recurring complaints: most negative themes by volume
complaints = (topic_df[topic_df["sentiment"] == "negative"]
              .groupby("theme").size().sort_values(ascending=False).rename("negative_sentences"))
complaints

## 3. Build the output table `review_nlp_results`
One row per review: overall sentiment + dominant topic. Linked back to source with `review_id`.

In [ ]:
from datetime import datetime

# review-level sentiment from the full review text
raw_rev = clf(reviews["text_light"].tolist())
rev_probs = pd.DataFrame([{d["label"]: d["score"] for d in row} for row in raw_rev])

results = reviews[["review_id"]].reset_index(drop=True).copy()
results["sentiment"] = rev_probs[["negative", "neutral", "positive"]].idxmax(axis=1)
results["sentiment_score"] = rev_probs["positive"] - rev_probs["negative"]

# dominant topic = the most frequent theme among the review's sentences
dominant = (topic_df.groupby("review_id")["theme"]
            .agg(lambda s: s.value_counts().index[0]).rename("topic_label"))
results = results.merge(dominant, on="review_id", how="left")
results["model_version"] = "distilbert-v1"
results["processed_at"] = datetime.now()
results.head()

In [ ]:
# save to a database table (SQLite here; use your own Postgres/MySQL URL in production)
engine = create_engine("sqlite:///data/outputs/montra_nlp.db")
results.to_sql("review_nlp_results", engine, if_exists="replace", index=False)
sent[["review_id", "sentence_id", "sentence", "topic_id", "theme", "sentiment", "sentiment_score"]] \
    .to_sql("sentence_nlp_results", engine, if_exists="replace", index=False)

# join back to the source data using review_id
pd.read_sql('''
    SELECT theme, sentiment, COUNT(*) AS n FROM sentence_nlp_results
    WHERE theme NOT IN ('outlier','other') GROUP BY theme, sentiment ORDER BY theme
''', engine).head()

### Dashboard idea
Connect Power BI / Tableau / Streamlit to `review_nlp_results` and `sentence_nlp_results`:
sentiment trend, topic x sentiment heatmap, hotel comparison, top recurring complaints.